# 11. 수작업 특징의 초기 unseen 실험

MusicGen과 Udio를 각각 Train·Validation에서 빼고 LR·SVM을 새로 학습한 과거 기록이다. 대상 FAKE와 원래 Test REAL을 비교해 처음 보는 생성기에서의 탐지를 확인한다. 이후 고정 설정 전이 실험은 26번에 별도로 있다.

In [2]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
)

PROJECT_ROOT = Path("/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project")

FEATURE_PATH = PROJECT_ROOT / "data/processed/features/handcrafted_features_10s.csv"

BASELINE_SUBGROUP_PATH = (
    PROJECT_ROOT / "results/baseline/subgroup_analysis/generator_metrics_all.csv"
)

RESULT_DIR = PROJECT_ROOT / "results/unseen_generator"
MODEL_DIR = PROJECT_ROOT / "checkpoints/unseen_generator"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

HOLDOUT_GENERATORS = ["musicgen", "udio"]
RANDOM_STATE = 42

print("FEATURE_PATH          :", FEATURE_PATH)
print("BASELINE_SUBGROUP_PATH:", BASELINE_SUBGROUP_PATH)
print("HOLDOUT_GENERATORS    :", HOLDOUT_GENERATORS)

FEATURE_PATH          : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/processed/features/handcrafted_features_10s.csv
BASELINE_SUBGROUP_PATH: /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/results/baseline/subgroup_analysis/generator_metrics_all.csv
HOLDOUT_GENERATORS    : ['musicgen', 'udio']


## 1. Feature 데이터 로드

In [3]:
df = pd.read_csv(FEATURE_PATH)

FEATURE_COLUMNS = sorted(
    [
        c
        for c in df.columns
        if (
            c.startswith("mfcc_")
            or c.startswith("mfcc_delta_")
            or c.startswith("mfcc_delta2_")
            or c.startswith("spectral_")
            or c.startswith("rms_")
            or c.startswith("zcr_")
        )
    ]
)

print("Rows:", len(df))
print("Feature columns:", len(FEATURE_COLUMNS))

print("\nSplit:")
# 같은 원곡의 표본이 여러 분할에 섞이지 않도록 저장된 split을 그대로 사용한다.
print(df["split"].value_counts())

print("\nFAKE generators:")
print(df.loc[df["label"] == "FAKE", "generator"].value_counts())

assert len(FEATURE_COLUMNS) == 266

Rows: 10077
Feature columns: 266

Split:
split
train    6967
test     1572
val      1538
Name: count, dtype: int64

FAKE generators:
generator
suno           900
udio           900
elevenlabs     900
diffrhythm     897
brev           894
acestep        882
musicgen       879
audioldm       876
songgen        582
stableaudio    582
producer       450
mubert         447
Name: count, dtype: int64


## 2. 평가 함수

In [4]:
# ROC에서 REAL 오탐과 FAKE 미탐의 균형을 확인한다.
def find_eer_threshold(y_true, scores):
    fpr, tpr, thresholds = roc_curve(
        y_true,
        scores,
        pos_label=1,
    )

    fnr = 1.0 - tpr
    valid = np.isfinite(thresholds)

    fpr = fpr[valid]
    fnr = fnr[valid]
    thresholds = thresholds[valid]

    idx = np.argmin(np.abs(fpr - fnr))

    return {
        "eer": float((fpr[idx] + fnr[idx]) / 2.0),
        "threshold": float(thresholds[idx]),
        "fpr_at_eer": float(fpr[idx]),
        "fnr_at_eer": float(fnr[idx]),
    }


def evaluate_scores(y_true, scores, threshold):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)

    y_pred = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    # EER은 두 오류율이 같아지는 지점이다. 분류에는 Validation에서 정한 임계값을 쓴다.
    eer_info = find_eer_threshold(y_true, scores)

    return {
        "n_total": len(y_true),
        "n_real": int((y_true == 0).sum()),
        "n_fake": int((y_true == 1).sum()),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "pr_auc": float(average_precision_score(y_true, scores)),
        "eer": float(eer_info["eer"]),
        "threshold_used": float(threshold),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "real_fpr": (float(fp / (fp + tn)) if (fp + tn) > 0 else np.nan),
        "fake_miss_rate": (float(fn / (fn + tp)) if (fn + tp) > 0 else np.nan),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def make_track_scores(meta_df, scores):
    temp = meta_df[
        [
            "track_sample_id",
            "original_audio",
            "label",
            "label_id",
            "genre",
            "generator",
            "split",
        ]
    ].copy()

    temp["score"] = scores

    return temp.groupby("track_sample_id", as_index=False).agg(
        original_audio=("original_audio", "first"),
        label=("label", "first"),
        label_id=("label_id", "first"),
        genre=("genre", "first"),
        generator=("generator", "first"),
        split=("split", "first"),
        segment_count=("score", "size"),
        score=("score", "mean"),
    )

## 3. Holdout 데이터셋 생성 함수

holdout generator는 Train / Validation에서 제거하고,
Test에서는 해당 generator의 FAKE와 REAL만 남긴다.

In [5]:
def build_holdout_data(df, holdout_generator):
    train = df[
        # 같은 원곡의 표본이 여러 분할에 섞이지 않도록 저장된 split을 그대로 사용한다.
        (df["split"] == "train")
        & (
            (df["label"] == "REAL")
            | ((df["label"] == "FAKE") & (df["generator"] != holdout_generator))
        )
    ].copy()

    val = df[
        (df["split"] == "val")
        & (
            (df["label"] == "REAL")
            | ((df["label"] == "FAKE") & (df["generator"] != holdout_generator))
        )
    ].copy()

    unseen_test = df[
        (df["split"] == "test")
        & (
            (df["label"] == "REAL")
            | ((df["label"] == "FAKE") & (df["generator"] == holdout_generator))
        )
    ].copy()

    seen_test = df[
        (df["split"] == "test")
        & (
            (df["label"] == "REAL")
            | ((df["label"] == "FAKE") & (df["generator"] != holdout_generator))
        )
    ].copy()

    return train, val, unseen_test, seen_test

## 4. Holdout 구조 QC

MusicGen과 Udio가 Train / Validation에서 실제로 0개가 되는지 먼저 확인한다.

In [6]:
# Holdout 구조 QC
holdout_structure_rows = []

for holdout in HOLDOUT_GENERATORS:
    train_h, val_h, unseen_h, seen_h = build_holdout_data(df, holdout)

    holdout_structure_rows.append(
        {
            "holdout_generator": holdout,
            "train_rows": len(train_h),
            "val_rows": len(val_h),
            "unseen_test_rows": len(unseen_h),
            "seen_test_rows": len(seen_h),
            "train_holdout_fake": int(
                ((train_h["label"] == "FAKE") & (train_h["generator"] == holdout)).sum()
            ),
            "val_holdout_fake": int(
                ((val_h["label"] == "FAKE") & (val_h["generator"] == holdout)).sum()
            ),
            "unseen_test_holdout_fake": int(
                (
                    (unseen_h["label"] == "FAKE") & (unseen_h["generator"] == holdout)
                ).sum()
            ),
            "train_original_audio": train_h["original_audio"].nunique(),
            "val_original_audio": val_h["original_audio"].nunique(),
            "test_original_audio": unseen_h["original_audio"].nunique(),
        }
    )

holdout_structure = pd.DataFrame(holdout_structure_rows)

display(holdout_structure)

,holdout_generator,train_rows,val_rows,unseen_test_rows,seen_test_rows,train_holdout_fake,val_holdout_fake,unseen_test_holdout_fake,train_original_audio,val_original_audio,test_original_audio
0,musicgen,6352,1409,270,1437,0,0,135,207,44,45
1,udio,6355,1394,279,1428,0,0,144,207,44,45


## 5. 단일 Holdout 실험 함수

각 holdout마다:

1. Train-only StandardScaler fit
2. Logistic Regression 학습
3. RBF-SVM 학습
4. Validation에서 segment / track EER threshold 결정
5. Unseen Test 평가
6. Seen-generator mixed Test도 참고용으로 평가

In [7]:
def run_holdout_experiment(holdout):
    train_df, val_df, unseen_test_df, seen_test_df = build_holdout_data(df, holdout)

    # ------------------------------
    # Feature arrays
    # ------------------------------
    X_train = train_df[FEATURE_COLUMNS].to_numpy(dtype=np.float64)
    X_val = val_df[FEATURE_COLUMNS].to_numpy(dtype=np.float64)
    X_unseen = unseen_test_df[FEATURE_COLUMNS].to_numpy(dtype=np.float64)
    X_seen = seen_test_df[FEATURE_COLUMNS].to_numpy(dtype=np.float64)

    y_train = train_df["label_id"].astype(int).to_numpy()
    y_val = val_df["label_id"].astype(int).to_numpy()
    y_unseen = unseen_test_df["label_id"].astype(int).to_numpy()
    y_seen = seen_test_df["label_id"].astype(int).to_numpy()

    # ------------------------------
    # Train-only scaling
    # ------------------------------
    # 특징마다 범위가 달라 표준화한다. 평균·표준편차는 Train에서만 구한다.
    scaler = StandardScaler()

    # Train의 평균·표준편차를 구하고 같은 Train 행을 변환한다.
    X_train_s = scaler.fit_transform(X_train)
    # Validation/Test에는 Train의 표준화 기준만 적용해 정보 누수를 막는다.
    X_val_s = scaler.transform(X_val)
    X_unseen_s = scaler.transform(X_unseen)
    # Validation/Test에는 Train의 표준화 기준만 적용해 정보 누수를 막는다.
    X_seen_s = scaler.transform(X_seen)

    joblib.dump(scaler, MODEL_DIR / f"{holdout}_scaler.joblib")

    result_rows = []
    prediction_outputs = {}

    model_specs = {
        "LogisticRegression": LogisticRegression(
            max_iter=3000,
            # 클래스별 표본 수가 달라 손실에 기여하는 비중을 조정한다.
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "RBF-SVM": SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            class_weight="balanced",
            probability=True,
            random_state=RANDOM_STATE,
            cache_size=2048,
        ),
    }

    for model_name, model in model_specs.items():
        print(f"Training {model_name} | holdout={holdout}")

        # fit은 모델 계수나 분류 경계를 학습한다. Validation/Test를 여기에 넣지 않는다.
        model.fit(X_train_s, y_train)

        joblib.dump(model, MODEL_DIR / f"{holdout}_{model_name}.joblib")

        # 확률 출력에서 모델이 예측한 클래스 순서에 맞는 점수 열을 사용한다.
        val_scores = model.predict_proba(X_val_s)[:, 1]
        unseen_scores = model.predict_proba(X_unseen_s)[:, 1]
        seen_scores = model.predict_proba(X_seen_s)[:, 1]

        # --------------------------
        # Segment threshold from Val
        # --------------------------
        # EER은 두 오류율이 같아지는 지점이다. 분류에는 Validation에서 정한 임계값을 쓴다.
        segment_val_eer = find_eer_threshold(y_val, val_scores)
        segment_threshold = segment_val_eer["threshold"]

        unseen_segment_metrics = evaluate_scores(
            y_unseen,
            unseen_scores,
            segment_threshold,
        )

        seen_segment_metrics = evaluate_scores(
            y_seen,
            seen_scores,
            segment_threshold,
        )

        result_rows.append(
            {
                "holdout_generator": holdout,
                "model": model_name,
                "level": "segment",
                "test_type": "unseen_generator",
                "val_eer": segment_val_eer["eer"],
                **unseen_segment_metrics,
            }
        )

        result_rows.append(
            {
                "holdout_generator": holdout,
                "model": model_name,
                "level": "segment",
                "test_type": "seen_generators",
                "val_eer": segment_val_eer["eer"],
                **seen_segment_metrics,
            }
        )

        # --------------------------
        # Track aggregation
        # --------------------------
        val_track = make_track_scores(val_df, val_scores)

        unseen_track = make_track_scores(unseen_test_df, unseen_scores)

        seen_track = make_track_scores(seen_test_df, seen_scores)

        track_val_eer = find_eer_threshold(
            val_track["label_id"],
            val_track["score"],
        )
        track_threshold = track_val_eer["threshold"]

        unseen_track_metrics = evaluate_scores(
            unseen_track["label_id"],
            unseen_track["score"],
            track_threshold,
        )

        seen_track_metrics = evaluate_scores(
            seen_track["label_id"],
            seen_track["score"],
            track_threshold,
        )

        result_rows.append(
            {
                "holdout_generator": holdout,
                "model": model_name,
                "level": "track",
                "test_type": "unseen_generator",
                "val_eer": track_val_eer["eer"],
                **unseen_track_metrics,
            }
        )

        result_rows.append(
            {
                "holdout_generator": holdout,
                "model": model_name,
                "level": "track",
                "test_type": "seen_generators",
                "val_eer": track_val_eer["eer"],
                **seen_track_metrics,
            }
        )

        # Save unseen predictions for later error analysis.
        unseen_segment_pred = unseen_test_df[
            [
                "segment_id",
                "track_sample_id",
                "original_audio",
                "label",
                "label_id",
                "genre",
                "generator",
                "split",
            ]
        ].copy()

        unseen_segment_pred["score"] = unseen_scores
        unseen_segment_pred["threshold"] = segment_threshold
        unseen_segment_pred["pred"] = (
            unseen_segment_pred["score"] >= segment_threshold
        ).astype(int)

        unseen_track_pred = unseen_track.copy()
        unseen_track_pred["threshold"] = track_threshold
        unseen_track_pred["pred"] = (
            unseen_track_pred["score"] >= track_threshold
        ).astype(int)

        prediction_outputs[(model_name, "segment")] = unseen_segment_pred

        prediction_outputs[(model_name, "track")] = unseen_track_pred

    return pd.DataFrame(result_rows), prediction_outputs

## 6. MusicGen / Udio Holdout 실행

RBF-SVM 때문에 약간 시간이 걸릴 수 있지만,
현재 데이터 크기에서는 일반적으로 오래 걸리지는 않는다.

In [8]:
# MusicGen / Udio Holdout 실행
all_results = []
all_predictions = {}

for holdout in HOLDOUT_GENERATORS:
    print("\n" + "=" * 80)
    print("HOLDOUT:", holdout)
    print("=" * 80)

    result_df, pred_dict = run_holdout_experiment(holdout)

    all_results.append(result_df)
    all_predictions[holdout] = pred_dict

unseen_results = pd.concat(
    all_results,
    ignore_index=True,
)

print("\nExperiment complete.")
print("Result rows:", len(unseen_results))


HOLDOUT: musicgen
Training LogisticRegression | holdout=musicgen
Training RBF-SVM | holdout=musicgen


/opt/anaconda3/envs/stat_env/lib/python3.11/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



HOLDOUT: udio
Training LogisticRegression | holdout=udio
Training RBF-SVM | holdout=udio


/opt/anaconda3/envs/stat_env/lib/python3.11/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Experiment complete.
Result rows: 16


## 7. Unseen-generator 전체 결과

먼저 `test_type = unseen_generator` 결과만 확인한다.

In [9]:
# Unseen-generator 전체 결과
unseen_only = unseen_results[unseen_results["test_type"] == "unseen_generator"].copy()

display(
    unseen_only[
        [
            "holdout_generator",
            "model",
            "level",
            "n_real",
            "n_fake",
            "roc_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
            "threshold_used",
        ]
    ]
    .sort_values(["holdout_generator", "level", "model"])
    .round(4)
)

,holdout_generator,model,level,n_real,n_fake,roc_auc,eer,balanced_accuracy,macro_f1,real_fpr,fake_miss_rate,threshold_used
0,musicgen,LogisticRegression,segment,135,135,0.5088,0.4667,0.5222,0.4965,0.2519,0.7037,0.6539
4,musicgen,RBF-SVM,segment,135,135,0.6046,0.4296,0.5111,0.4559,0.1704,0.8074,0.9189
2,musicgen,LogisticRegression,track,45,45,0.5309,0.5000,0.4889,0.4250,0.1778,0.8444,0.6541
6,musicgen,RBF-SVM,track,45,45,0.6123,0.4000,0.5222,0.4669,0.1556,0.8000,0.8702
8,udio,LogisticRegression,segment,135,144,0.7325,0.2977,0.6870,0.6837,0.2370,0.3889,0.6651
12,udio,RBF-SVM,segment,135,144,0.8102,0.2868,0.7243,0.7178,0.1556,0.3958,0.9202
10,udio,LogisticRegression,track,45,48,0.7968,0.2146,0.6833,0.6688,0.1333,0.5000,0.6675
14,udio,RBF-SVM,track,45,48,0.8542,0.2368,0.7361,0.7266,0.1111,0.4167,0.8890


## 8. 핵심 결과 — RBF-SVM Track-level Unseen Generator

현재 baseline에서 가장 강한 설정인
RBF-SVM + Track aggregation 결과를 우선 확인한다.

In [10]:
# 핵심 결과 — RBF-SVM Track-level Unseen Generator
svm_track_unseen = unseen_only[
    (unseen_only["model"] == "RBF-SVM") & (unseen_only["level"] == "track")
].copy()

display(
    svm_track_unseen[
        [
            "holdout_generator",
            "n_real",
            "n_fake",
            "roc_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
            "threshold_used",
        ]
    ].round(4)
)

,holdout_generator,n_real,n_fake,roc_auc,eer,balanced_accuracy,macro_f1,real_fpr,fake_miss_rate,threshold_used
6,musicgen,45,45,0.6123,0.4000,0.5222,0.4669,0.1556,0.8000,0.8702
14,udio,45,48,0.8542,0.2368,0.7361,0.7266,0.1111,0.4167,0.8890


## 9. Segment-level vs Track-level 비교

In [11]:
# Segment-level vs Track-level 비교
svm_unseen_compare = unseen_only[unseen_only["model"] == "RBF-SVM"].pivot(
    index="holdout_generator",
    columns="level",
    values=[
        "roc_auc",
        "eer",
        "balanced_accuracy",
        "fake_miss_rate",
    ],
)

display(svm_unseen_compare.round(4))

roc_auc             eer         balanced_accuracy          \
level             segment   track segment   track           segment   track   
holdout_generator                                                             
musicgen           0.6046  0.6123  0.4296  0.4000            0.5111  0.5222   
udio               0.8102  0.8542  0.2868  0.2368            0.7243  0.7361   

                  fake_miss_rate          
level                    segment   track  
holdout_generator                         
musicgen                  0.8074  0.8000  
udio                      0.3958  0.4167

## 10. In-domain vs Unseen-generator 비교

10단계 subgroup 분석의 in-domain generator 성능과 비교한다.

- In-domain:
  해당 generator를 Train에서 본 상태
- Unseen:
  해당 generator를 Train / Validation에서 완전히 제거한 상태

성능 하락 폭이 **generator generalization gap**이다.

In [12]:
if BASELINE_SUBGROUP_PATH.exists():
    baseline_subgroup = pd.read_csv(BASELINE_SUBGROUP_PATH)

    in_domain = baseline_subgroup[
        (baseline_subgroup["model"] == "RBF-SVM")
        & (baseline_subgroup["level"] == "track")
        & (baseline_subgroup["generator"].isin(HOLDOUT_GENERATORS))
    ][
        [
            "generator",
            "roc_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
        ]
    ].copy()

    in_domain = in_domain.rename(
        columns={
            "generator": "holdout_generator",
            "roc_auc": "in_domain_roc_auc",
            "eer": "in_domain_eer",
            "balanced_accuracy": "in_domain_balanced_accuracy",
            "macro_f1": "in_domain_macro_f1",
            "real_fpr": "in_domain_real_fpr",
            "fake_miss_rate": "in_domain_fake_miss_rate",
        }
    )

    unseen_compare = svm_track_unseen.merge(
        in_domain,
        on="holdout_generator",
        how="left",
        validate="one_to_one",
    )

    unseen_compare["roc_auc_drop"] = (
        unseen_compare["in_domain_roc_auc"] - unseen_compare["roc_auc"]
    )

    unseen_compare["eer_increase"] = (
        unseen_compare["eer"] - unseen_compare["in_domain_eer"]
    )

    unseen_compare["fake_miss_increase"] = (
        unseen_compare["fake_miss_rate"] - unseen_compare["in_domain_fake_miss_rate"]
    )

    display(
        unseen_compare[
            [
                "holdout_generator",
                "in_domain_roc_auc",
                "roc_auc",
                "roc_auc_drop",
                "in_domain_eer",
                "eer",
                "eer_increase",
                "in_domain_fake_miss_rate",
                "fake_miss_rate",
                "fake_miss_increase",
            ]
        ].round(4)
    )
else:
    print("Baseline subgroup result file not found:", BASELINE_SUBGROUP_PATH)

,holdout_generator,in_domain_roc_auc,roc_auc,roc_auc_drop,in_domain_eer,eer,eer_increase,in_domain_fake_miss_rate,fake_miss_rate,fake_miss_increase
0,musicgen,0.9951,0.6123,0.3827,0.0556,0.4000,0.3444,0.0222,0.8000,0.7778
1,udio,0.9014,0.8542,0.0472,0.2042,0.2368,0.0326,0.3125,0.4167,0.1042


## 11. Seen-generator Mixed Test 참고 결과

holdout generator를 제외한 나머지 generator에서는
모델이 여전히 정상적으로 작동하는지 확인한다.

이 결과는 주 실험이 아니라 sanity check 용도다.

In [13]:
# Seen-generator Mixed Test 참고 결과
seen_results = unseen_results[unseen_results["test_type"] == "seen_generators"]

display(
    seen_results[
        [
            "holdout_generator",
            "model",
            "level",
            "roc_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
        ]
    ]
    .sort_values(["holdout_generator", "level", "model"])
    .round(4)
)

,holdout_generator,model,level,roc_auc,eer,balanced_accuracy,macro_f1,real_fpr,fake_miss_rate
1,musicgen,LogisticRegression,segment,0.8744,0.2180,0.7884,0.6668,0.2519,0.1713
5,musicgen,RBF-SVM,segment,0.9201,0.1580,0.8480,0.7259,0.1704,0.1336
3,musicgen,LogisticRegression,track,0.9095,0.1936,0.8398,0.7107,0.1778,0.1425
7,musicgen,RBF-SVM,track,0.9591,0.1468,0.8677,0.7552,0.1556,0.1091
9,udio,LogisticRegression,segment,0.8892,0.1941,0.8038,0.6864,0.2370,0.1555
13,udio,RBF-SVM,segment,0.9376,0.1506,0.8623,0.7463,0.1556,0.1199
11,udio,LogisticRegression,track,0.9334,0.1171,0.8706,0.7418,0.1333,0.1256
15,udio,RBF-SVM,track,0.9760,0.1105,0.8873,0.7610,0.1111,0.1143


## 12. Unseen Test Prediction 저장

향후 error analysis와 CNN 비교를 위해
holdout generator의 prediction을 저장한다.

In [14]:
for holdout, pred_dict in all_predictions.items():
    holdout_dir = RESULT_DIR / holdout
    holdout_dir.mkdir(parents=True, exist_ok=True)

    for (model_name, level), pred_df in pred_dict.items():
        safe_model = model_name.lower().replace("-", "_")

        pred_df.to_csv(
            holdout_dir / f"{safe_model}_{level}_unseen_predictions.csv",
            index=False,
            encoding="utf-8-sig",
        )

print("Saved unseen prediction files.")

Saved unseen prediction files.


## 13. Metrics 저장

In [15]:
unseen_results.to_csv(
    RESULT_DIR / "unseen_generator_metrics_all.csv",
    index=False,
    encoding="utf-8-sig",
)

svm_track_unseen.to_csv(
    RESULT_DIR / "rbf_svm_track_unseen_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

if "unseen_compare" in globals():
    unseen_compare.to_csv(
        RESULT_DIR / "in_domain_vs_unseen_comparison.csv",
        index=False,
        encoding="utf-8-sig",
    )

print("Saved metrics to:", RESULT_DIR)

Saved metrics to: /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/results/unseen_generator


## 14. 최종 QC

핵심 검증:

- holdout generator는 Train에서 0개
- holdout generator는 Validation에서 0개
- Test에는 holdout FAKE가 존재
- MusicGen / Udio 모두 실험 완료
- ROC-AUC / threshold 값이 정상적으로 계산됨

In [16]:
# 최종 QC
qc_rows = []

for holdout in HOLDOUT_GENERATORS:
    structure = holdout_structure[
        holdout_structure["holdout_generator"] == holdout
    ].iloc[0]

    subset = unseen_only[unseen_only["holdout_generator"] == holdout]

    qc_rows.append(
        {
            "holdout_generator": holdout,
            "train_holdout_fake": int(structure["train_holdout_fake"]),
            "val_holdout_fake": int(structure["val_holdout_fake"]),
            "unseen_test_holdout_fake": int(structure["unseen_test_holdout_fake"]),
            "result_rows": len(subset),
            "nan_roc_auc": int(subset["roc_auc"].isna().sum()),
            "nonfinite_threshold": int((~np.isfinite(subset["threshold_used"])).sum()),
        }
    )

qc_summary = pd.DataFrame(qc_rows)

display(qc_summary)

unseen_qc_pass = (
    (qc_summary["train_holdout_fake"] == 0).all()
    and (qc_summary["val_holdout_fake"] == 0).all()
    and (qc_summary["unseen_test_holdout_fake"] > 0).all()
    and (qc_summary["result_rows"] == 4).all()
    and (qc_summary["nan_roc_auc"] == 0).all()
    and (qc_summary["nonfinite_threshold"] == 0).all()
)

print("===== FINAL RESULT =====")
print("Unseen Generator Core QC PASS:", unseen_qc_pass)

,holdout_generator,train_holdout_fake,val_holdout_fake,unseen_test_holdout_fake,result_rows,nan_roc_auc,nonfinite_threshold
0,musicgen,0,0,135,4,0,0
1,udio,0,0,144,4,0,0


===== FINAL RESULT =====
Unseen Generator Core QC PASS: True


## 이어지는 기록

같은 모델이 MP3 압축에 얼마나 민감한지는 12번에서 비교한다.

## 초기 생성기 제외 결과

RBF-SVM Track-level unseen-generator 결과다.

| Holdout generator | ROC-AUC | EER | Balanced Accuracy | FAKE Miss Rate |
|---|---:|---:|---:|---:|
| MusicGen | 0.6123 | 0.4000 | 0.5222 | 0.8000 |
| Udio | 0.8542 | 0.2368 | 0.7361 | 0.4167 |

- Holdout FAKE는 Train/Validation에서 완전히 제외한 상태로 평가했다.
- 특히 MusicGen은 in-domain 대비 큰 일반화 하락을 보여 generator shift의 영향을 확인했다.
- 결과를 `results/unseen_generator/`에 저장했다.